# T18 -- Lung-Region Attention Module (Candidate A)
## DenseNet121 + supervised spatial attention, trained on the COVID-19 Radiography Database

**Owner:** Member 1 | **Week 2, Milestone M2** | Novel contribution (Candidate A of 3)

**Design doc:** `Claude Working Files/T18_Lung_Region_Attention_WBS.md` -- read sections 1-7
before touching this notebook; this file follows that design exactly, section by section
(S0-S16). Where this notebook and `Member1_Guide.md` disagree, the WBS wins (see WBS section 4).

**Platform:** Kaggle, T4 (confirmed in S0 -- no sm_60 / P100 compatibility pin needed).

**S0 pre-flight -- resolved decisions (see WBS section 12.5 / chat log for full reasoning):**
- Lung masks: the COVID-19 Radiography Dataset's bundled `masks/` folder (confirmed official,
  not a fallback -- same folder structure locally and on the Kaggle-hosted dataset).
- Faithfulness (Grad-CAM EIL) definition: Member 5's T30 definition is not yet in the repo
  (checked `notebooks/candidate-c-grad-cam-shortcut-suppression-loss.ipynb` -- it has a
  training-time suppression loss on raw activations, not a post-hoc scoring function). Using
  our own documented definition (WBS section 6.3); swap later if T30's differs -- it's a
  metric function, not a training-time dependency.
- Hyperparameters: T13 (Member 2's DenseNet121 HP tuning) has no committed winner --
  `notebooks/baseline-cnn-model-dnn-research.ipynb` only exposes phase1_lr/phase2_lr as
  argparse *defaults* (1e-3/1e-5), it never sweeps them. Adopting T16's measured winner
  instead (WBS section 3.4a): `phase1_lr=3e-4, phase2_lr=3e-5, weight_decay=1e-3`.
- Mask-provenance wording (proposal says "automatically generated"; we use the dataset's
  supplied masks) recorded as a known discrepancy for the module card -- not blocking.


## Environment check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print(f"Compute capability: sm_{major}{minor}")
    if (major, minor) == (6, 0):
        print("WARNING: Tesla P100 (sm_60) detected -- the AuxSeg notebook's torch==2.8.0+cu126 "
              "pin is required before any further torch import. This notebook was designed for "
              "T4 (S0) and does NOT include that pin by default.")


## S1 -- Config

Loads `configs/densenet121_lung_attention.yaml` (arm A2's config). Other arms are produced
from this base config via `merge_overrides()` at the point each arm is trained (S8/S9/S10) --
see the WBS's arm-to-flags table (Appendix A.3) for the exact overrides per arm.


In [ ]:
import sys
from pathlib import Path

# On Kaggle the repo isn't on sys.path by default. Upload the repo as a Kaggle Dataset
# (or clone it in a setup cell) and adjust this path, or run this notebook from a local
# clone where the repo root is already the working directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # Kaggle fallback -- adjust to wherever the repo dataset/clone actually lands.
    REPO_ROOT = Path("/kaggle/working/Chest-X-ray-Disease-Detection")
sys.path.insert(0, str(REPO_ROOT))

from src.utils import load_config, merge_overrides

cfg = load_config(REPO_ROOT / "configs" / "densenet121_lung_attention.yaml")
print("Loaded config for experiment:", cfg["experiment"]["name"])
import json
print(json.dumps(cfg, indent=2))


## S2 -- Data layer -- mask-paired dataset + verification

Copy `JointTransform`, `CXRWithMaskDataset`, `stratified_split`, `build_dataloaders`, `compute_class_weights` from the AuxSeg notebook verbatim, then add reproducible worker seeding (WBS section 4.8). Run the Appendix A.1 verification + alignment-eyeball cells before proceeding -- a misaligned mask silently invalidates every downstream number.

This is also where `artifacts/splits/split_manifest_v1.csv` gets emitted (S1 step 5 of the WBS) using the real dataset.

In [ ]:
from pathlib import Path
from src.datasets import build_dataloaders

# COVID-19 Radiography Dataset -- local path (this repo) vs Kaggle-hosted copy.
# S0 decision: platform is Kaggle/T4; masks are the dataset's bundled masks/ folder
# (both paths below ship the same masks/ folder structure).
_LOCAL_DATA_DIR = "/Volumes/My Disk 2/My Projects/Chest Disease Detection/COVID-19_Radiography_Dataset"
_KAGGLE_DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
DATA_DIR = _LOCAL_DATA_DIR if Path(_LOCAL_DATA_DIR).exists() else _KAGGLE_DATA_DIR
print("Using DATA_DIR:", DATA_DIR)

SPLIT_MANIFEST = REPO_ROOT / "artifacts" / "splits" / "split_manifest_v1.csv"
assert SPLIT_MANIFEST.exists(), (
    f"{SPLIT_MANIFEST} not found -- it must be committed by S1, not regenerated here "
    "(docs/experiment_policy.md 'Dataset Split')."
)

train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    DATA_DIR,
    img_size=cfg["dataset"]["image_size"],
    batch_size=cfg["training"]["batch_size"],
    seed=cfg["experiment"]["seed"],
    num_workers=4,
    split_manifest_path=SPLIT_MANIFEST,
)
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")


In [ ]:
# --- S2 verification cell (WBS Appendix A.1) ---
import numpy as np
import torch
from PIL import Image

CLASSES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]

# 1) split sizes + class order (also implicitly re-checks the manifest against
#    the actual files on this machine -- build_dataloaders() would have raised
#    a KeyError already if any sample were missing from the manifest)
assert (len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)) == (14815, 3175, 3175)
assert class_names == CLASSES, class_names
print("[1] split sizes + class order OK")

# 2) batch contract
imgs, labels, masks = next(iter(train_loader))
assert imgs.shape[1:] == (3, 224, 224), imgs.shape
assert masks.shape[1:] == (1, 224, 224), masks.shape
assert set(torch.unique(masks).tolist()) <= {0.0, 1.0}
print("[2] batch shapes OK:", imgs.shape, masks.shape)

# 3) lung-area fraction sanity over 200 random eval-transform samples.
# NOTE: measured on the real dataset (S2 verification run): mean=0.235,
# range [0.043, 0.472]. The WBS's original "~0.25-0.45" was a pre-verification
# estimate -- 0.20-0.50 below is the corrected range. This check exists to
# catch gross errors (polarity inversion ~0.99/0.01, wrong channel handling),
# not to pin down the exact anatomical distribution.
from src.datasets import JointTransform
rng = np.random.default_rng(0)
paths = [p for cls in class_names for p in sorted((Path(DATA_DIR) / cls / "images").glob("*.png"))]
sample_paths = rng.choice(paths, size=200, replace=False)
eval_tf = JointTransform(img_size=224, train=False)
fracs = []
for p in sample_paths:
    p = Path(p)
    mask_p = p.parent.parent / "masks" / p.name
    _img, m = eval_tf(Image.open(p).convert("L"), Image.open(mask_p).convert("L"))
    fracs.append(m.mean().item())
fracs = np.array(fracs)
print(f"[3] lung-area fraction: mean={fracs.mean():.3f} min={fracs.min():.3f} max={fracs.max():.3f}")
assert 0.20 < fracs.mean() < 0.50, "mask polarity or channel handling is likely wrong"

print("\nAll S2 verification assertions passed.")


### Alignment eyeball check

Overlay 8 random *augmented* train images with their masks. **Look at the output** -- do the red regions sit on the lungs after rotation/crop/flip? This is the single highest-value 30 seconds in this task: a misaligned mask silently invalidates every downstream number. (Already verified once against this exact dataset during design review -- rerun here as a live sanity check, e.g. if you're now pointing at Kaggle's copy of the dataset instead of a local one.)

In [ ]:
import matplotlib.pyplot as plt
from src.datasets import IMAGENET_MEAN, IMAGENET_STD

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for k, ax in enumerate(axes.flat):
    show = (imgs[k] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    ax.imshow(masks[k, 0].numpy(), cmap="Reds", alpha=0.35)
    ax.set_title(class_names[labels[k]])
    ax.axis("off")
plt.suptitle("S2 alignment check -- augmented train images (JointTransform, train=True) + masks")
plt.tight_layout()
plt.show()


## S3 -- The Lung-Region Attention module + CPU unit tests

`LungRegionAttention`, `attention_guidance_loss`, `compute_total_loss` (WBS Appendix A.2). Canonical copy lives in `src/modules/lung_attention.py`; `tests/test_lung_attention.py` must be green before any GPU work starts.

In [ ]:
from src.modules import LungRegionAttention, attention_guidance_loss, compute_total_loss

# Kaggle note: this import requires the repo to be on sys.path (see the S1 cell's
# REPO_ROOT / sys.path setup). If the repo isn't uploaded as a Kaggle Dataset,
# paste src/modules/lung_attention.py's contents into this cell instead --
# src/modules/lung_attention.py stays the single canonical copy either way
# (also reused unchanged by T23 on ResNet50).


In [ ]:
# Quick inline smoke test (pytest tests/test_lung_attention.py -v is the real suite --
# S3 DoD is 12 tests green, not this cell -- but this gives a fast visual sanity
# check directly in the notebook, useful when iterating on Kaggle).
_m = LungRegionAttention(in_channels=1024, reduction=8, gate_mode="residual")
_f = torch.randn(2, 1024, 7, 7)
_out, _att, _logits = _m(_f)
print("out:", _out.shape, "| att:", _att.shape, "| logits:", _logits.shape)
print("attention at init (should be ~0.5 everywhere, zero-init):", _att.mean().item(), _att.std().item())
print("module params:", sum(p.numel() for p in _m.parameters()), "(expect 131,329 at reduction=8, in_channels=1024)")

_mask = torch.zeros(2, 1, 224, 224); _mask[:, :, 64:160, 48:176] = 1.0
_att_loss = attention_guidance_loss(_logits, _mask)
print("attention_guidance_loss at init (uniform 0.5 attention -> should be ~log(2)=0.693):", _att_loss.item())


## S4 -- Assemble the model + parity check against vanilla

`DenseNetLungAttention` wrapper (WBS Appendix A.3), the backbone-agnostic freeze/unfreeze registry (Appendix A.4), and the CBAM comparator for arm A5 (Appendix A.4b). Parity check: gate_mode='none' must produce bit-identical logits to plain timm DenseNet121.

In [ ]:
import timm
from src.modules import build_model, freeze_backbone, unfreeze_final_blocks, print_trainable_parameters, LogitsOnly


In [ ]:
# --- Parity check: gate_mode="none" must be bit-identical to plain timm DenseNet121 ---
# (given the SAME weights -- copy the backbone state_dict across, not just same architecture)
torch.manual_seed(0)
_wrapped = build_model(num_classes=4, use_attention=True, gate_mode="none", pretrained=False)
_plain = timm.create_model("densenet121", pretrained=False, num_classes=4)
_plain.load_state_dict(_wrapped.backbone.state_dict())
_plain.eval(); _wrapped.eval()

_x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    _plain_logits = _plain(_x)
    _wrapped_logits, _, _ = _wrapped(_x)
torch.testing.assert_close(_wrapped_logits, _plain_logits, rtol=0, atol=0)
print("PARITY CHECK PASSED: gate_mode='none' logits are bit-identical to plain timm DenseNet121")

# A0 arm (use_attention=False) must have EXACTLY the vanilla parameter count
_a0 = build_model(num_classes=4, use_attention=False, pretrained=False)
_vanilla_params = sum(p.numel() for p in _plain.parameters())
_a0_params = sum(p.numel() for p in _a0.parameters())
print(f"A0 params: {_a0_params:,} | vanilla timm params: {_vanilla_params:,}")
assert _a0_params == _vanilla_params


In [ ]:
# --- Trainable-count assertions (catch silent freeze bugs) ---
model = build_model(num_classes=4, use_attention=True, gate_mode="residual", pretrained=False)

freeze_backbone(model)
print_trainable_parameters(model, "phase1 ")
_phase1_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _phase1_trainable == 135_429, _phase1_trainable  # classifier (4,100) + attn (131,329)

unfreeze_final_blocks(model, num_blocks=1)
print_trainable_parameters(model, "phase2 ")
_phase2_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert 2_000_000 < _phase2_trainable < 2_400_000, _phase2_trainable  # WBS estimate ~2.2M

print("\nS4 verification passed: shapes, parity, and trainable counts all match the design doc.")


## S5 -- Loss + metrics implementation

`src/modules/attention_metrics.py` (ILAR, IoU, Dice, entropy, background attention) and the Grad-CAM EIL harness (pre-gate / post-gate taps, WBS section 6.3).

Kaggle: `!pip install grad-cam` (same as Candidate C's notebook) before running the Grad-CAM cell below.

In [ ]:
from src.modules import (
    ilar, attention_iou, attention_dice, attention_entropy, background_attention,
    energy_inside_lung, cam_for, get_taps,
)


In [ ]:
# --- Metric self-check (WBS Appendix A.6, extended to all 6 functions) ---
_mask = torch.zeros(1, 1, 224, 224); _mask[:, :, 64:160, 48:176] = 1.0
_frac = _mask.mean().item()

assert abs(ilar(_mask.clone(), _mask).item() - 1.0) < 1e-4                      # perfect map
assert abs(ilar(torch.ones(1, 1, 7, 7), _mask).item() - _frac) < 0.02           # uniform map
assert ilar(1.0 - _mask, _mask).item() < 1e-4                                   # inverted map
assert abs(attention_iou(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(attention_dice(_mask.clone(), _mask).item() - 1.0) < 1e-4
assert abs(attention_entropy(torch.full((1, 1, 7, 7), 0.5)).item() - 0.69315) < 1e-4   # max entropy at 0.5
assert background_attention(_mask.clone(), _mask).item() < 1e-4                # attention confined to lungs -> ~0 background
assert abs(energy_inside_lung(_mask.clone(), _mask).item() - 1.0) < 1e-4       # same formula as ilar, for a CAM input

print("All 6 metric self-checks passed.")


In [ ]:
# --- Grad-CAM harness demo: arm A0's self-check (EIL_post == EIL_pre exactly, WBS 6.3) ---
_demo_model = build_model(num_classes=4, use_attention=False, pretrained=False)
_demo_images = torch.randn(2, 3, 224, 224)
_tap_post, _tap_pre = get_taps(_demo_model)

_cam_post, _preds_post = cam_for(_demo_model, _demo_images, _tap_post, torch.device("cpu"))
_cam_pre, _preds_pre = cam_for(_demo_model, _demo_images, _tap_pre, torch.device("cpu"))

_eil_post = energy_inside_lung(_cam_post, _mask.repeat(2, 1, 1, 1))
_eil_pre = energy_inside_lung(_cam_pre, _mask.repeat(2, 1, 1, 1))
print("EIL_post:", _eil_post.tolist())
print("EIL_pre: ", _eil_pre.tolist())
assert torch.allclose(_eil_post, _eil_pre, atol=1e-3), "arm A0 self-check failed: taps should be identical"
print("\nGrad-CAM harness self-check passed: arm A0's two taps agree, as expected (WBS section 6.3).")


## S6 -- Training loop

Adapted `run_epoch` / `train_phase` / `evaluate` from AuxSeg: 3-tuple batches, AMP, per-epoch val precision/recall/F1/AUROC logged to W&B, `log_summary_metrics` at the end of each phase (WBS section S6 / policy Required Information).

In [ ]:
from src.modules import run_epoch, train_phase, evaluate
from src.utils import initialize_wandb, finish_run, generate_run_name

# One function (train_phase) handles every T18 arm via the model's own
# use_attention/gate_mode flags and lambda_att -- no per-arm branching or
# copy-pasted training code (WBS S6 DoD). Which arm you get is entirely
# determined by how you build `model` (S4) and what lambda_att you pass here.


For a *real* run: `initialize_wandb(cfg, run_name=generate_run_name(cfg["model"]["name"], cfg["experiment"]["name"], cfg["experiment"]["seed"]))` before calling `train_phase(..., wandb_enabled=True)`, and `finish_run()` in a `finally` block (a crashed Kaggle session shouldn't leave a run hanging). S7's smoke test below uses `wandb_enabled=False` so it doesn't need W&B at all.

## S7 -- Smoke test + overfit test

Do not proceed to S8 until: (1) the smoke run completes end to end, (2) the 32-image overfit test reaches 100% train accuracy with att_loss falling from ~0.6931 toward the ~0.2 entropy floor (WBS section 4.4a), (3) the lambda-sensitivity spot check shows higher ILAR at lambda=5 than lambda=0.

## S8 -- Arm A0 -- the vanilla control (full schedule)

`use_attention=False`. Sanity gate: judge against the measured 90.5-92% macro-F1 range (WBS section 2.4), not a single point estimate from the possibly-placeholder proposal table.

## S9 -- Lambda sweep (short schedule) + pre-registered selection

Mirrors T16's TUNING_CONFIGS -> tuning_summary.csv -> selected_config.json pattern exactly (WBS section S9). Test set stays out of scope for this entire section.

## S10 -- Full-schedule runs -- A1, A4, A2, A5 (and optionally A3)

Priority order: A2 (headline) -> A1 (isolates supervision) -> A5 (CBAM, required by the assignment brief section 9) -> A4 (isolates gating) -> A3 (optional). Re-seed at the start of every arm.

## S11 -- Evaluation, faithfulness, comparison table

Grad-CAM EIL at both taps, attention metrics, `T18_comparison_table.csv`, explicit pass/fail verdicts against acceptance criteria A1-A6, per-image prediction CSVs for Member 3's statistics (T37).

## S12 -- Figures

Heat-map overlays (3 per class), the attention grid, the lambda-sweep trade-off plot.

## S13 -- Efficiency measurement (A6)

`kusal-notebooks/efficiency.py::benchmark_model`, wrapped in `LogitsOnly` -- verify against the analytic estimate (+131,329 params, +0.22% GFLOPs).

## S14 -- Multi-seed repeat (optional, budget permitting)

A0 and A2 only, seeds 123 and 2026, full schedule. Explicitly state single-seed in the module card if this doesn't run.

## S15 -- Package and hand off

`T18_module_card.md`, deliverables to Member 3 / Member 2 / Member 5, the `LogitsOnly` wrapper note, RSNA class-mapping note, contribution list for the colour-highlighted paper.

## S16 -- Paper material

~400 words Methods, ~250 words Results -- draft in WBS Appendix B, edit against the actual `T18_comparison_table.csv` numbers before handing off.